# Deep Q-Networks
## DQN, Double DQN, and Dueling DQN: From Scratch in PyTorch

**What You'll Learn:**
- Experience replay buffers and why they're needed
- Target networks for training stability
- $\varepsilon$-decay schedules for exploration
- DQN architecture and training loop
- Double DQN: decoupling action selection and evaluation
- Dueling DQN: separate value and advantage streams
- Solving CartPole-v1 with each variant

**Prerequisites:** Function approximation (Notebook 5), basic neural networks, PyTorch fundamentals.

**References:**
- Mnih et al., *Human-level control through deep reinforcement learning*, Nature 2015 (DQN)
- van Hasselt et al., *Deep Reinforcement Learning with Double Q-learning*, AAAI 2016 (Double DQN)
- Wang et al., *Dueling Network Architectures for Deep Reinforcement Learning*, ICML 2016 (Dueling DQN)

---
## 1. From Tabular to Deep Q-Learning

In tabular Q-learning we maintain a table $Q(s, a)$ for every state-action pair. This is infeasible
when the state space is continuous or very large. The idea behind **Deep Q-Networks** is to
replace the table with a neural network parameterised by $\theta$:

$$Q(s, a) \approx Q(s, a;\, \theta)$$

The network takes a state $s$ as input and outputs Q-values for **all** actions simultaneously.

### Why naive function approximation fails

Two problems arise when we simply plug a neural network into Q-learning:

1. **Correlated samples** — Consecutive transitions $(s_t, a_t, r_t, s_{t+1})$ are highly correlated.
   SGD assumes i.i.d. samples, so learning becomes unstable.

2. **Non-stationary targets** — The target $r + \gamma \max_{a'} Q(s', a';\, \theta)$ depends on the
   same parameters $\theta$ we are updating, creating a moving-target problem.

### Solutions introduced by DQN

| Problem | Solution |
|---|---|
| Correlated samples | **Experience Replay** — store transitions in a buffer and sample uniformly |
| Non-stationary targets | **Target Network** — a frozen copy $\theta^-$ updated periodically |

---
## 2. DQN Loss Function

The DQN agent minimises the mean squared Bellman error over mini-batches drawn from the
replay buffer $\mathcal{D}$:

$$\boxed{L(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[ \left( r + \gamma \max_{a'} Q(s', a';\, \theta^-) - Q(s, a;\, \theta) \right)^2 \right]}$$

where $\theta^-$ denotes the **target network** parameters. These are copied from the online
network every $C$ episodes (or steps), keeping the target stable between updates.

---
## 3. Double DQN

Standard DQN uses $\max_{a'} Q(s', a';\, \theta^-)$ as the target. Because the $\max$ operator
both **selects** and **evaluates** using the same values, it tends to **overestimate** Q-values.

**Double DQN** decouples selection from evaluation:

- **Select** the best action using the **online** network $\theta$
- **Evaluate** that action using the **target** network $\theta^-$

$$\boxed{Y^{\text{DDQN}} = r + \gamma\, Q\!\left(s',\, \arg\max_{a'} Q(s', a';\, \theta);\, \theta^-\right)}$$

This simple change significantly reduces overestimation bias and often leads to better policies.

---
## 4. Dueling DQN

The **Dueling** architecture separates the Q-function into two streams:

- **Value stream** $V(s;\, \theta_V)$ — how good is this state?
- **Advantage stream** $A(s, a;\, \theta_A)$ — how much better is action $a$ than average?

They are combined via:

$$\boxed{Q(s,a;\,\theta) = V(s;\,\theta_V) + A(s,a;\,\theta_A) - \frac{1}{|\mathcal{A}|}\sum_{a'} A(s,a';\,\theta_A)}$$

Subtracting the mean advantage ensures **identifiability**: the value stream is forced to
capture the true state value, while the advantage stream captures action-specific deviations.

This architecture is especially useful in states where the choice of action does not matter
much (e.g., no immediate danger), because $V$ can be learned without needing to evaluate
every action.

---
## 5. Setup and Constants

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import deque, namedtuple
import random
import warnings
warnings.filterwarnings('ignore')

# ---------- reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------- plotting ----------
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})
COLORS = ['steelblue', 'coral', 'seagreen', 'goldenrod', 'mediumpurple']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# ==================== Hyperparameters ====================
GAMMA         = 0.99      # discount factor
LR            = 1e-3      # learning rate
BATCH_SIZE    = 64        # mini-batch size for replay sampling
BUFFER_SIZE   = 10_000    # replay buffer capacity
TARGET_UPDATE = 10        # copy online -> target every N episodes
EPS_START     = 1.0       # initial exploration rate
EPS_END       = 0.01      # minimum exploration rate
EPS_DECAY     = 0.995     # multiplicative decay per episode
N_EPISODES    = 500       # total training episodes
HIDDEN_DIM    = 128       # neurons per hidden layer

# named tuple for clean transition storage
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

print("Hyperparameters set.")

---
## 6. Experience Replay Buffer

The replay buffer stores transitions and provides uniform random mini-batches.
This breaks temporal correlations and allows each transition to be reused many times.

In [ ]:
class ReplayBuffer:
    """Fixed-size replay buffer using a deque."""

    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        """Store a single transition."""
        self.buffer.append(Transition(state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        """Sample a random mini-batch and return tensors on `device`."""
        transitions = random.sample(self.buffer, batch_size)
        batch = Transition(*zip(*transitions))

        states      = torch.FloatTensor(np.array(batch.state)).to(device)
        actions     = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        rewards     = torch.FloatTensor(batch.reward).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones       = torch.FloatTensor(batch.done).unsqueeze(1).to(device)

        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)


# Quick sanity check
buf = ReplayBuffer(capacity=100)
for i in range(150):  # push more than capacity
    buf.push(np.zeros(4), 0, 1.0, np.zeros(4), False)
assert len(buf) == 100, "Buffer should cap at capacity"
print(f"Buffer length after 150 pushes (capacity 100): {len(buf)}")
print("ReplayBuffer sanity check passed.")

---
## 7. Network Architectures

In [ ]:
class DQNNetwork(nn.Module):
    """Standard DQN: two hidden layers with ReLU activations."""

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = HIDDEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        return self.net(x)


class DuelingDQNNetwork(nn.Module):
    """Dueling DQN: shared features split into value and advantage streams."""

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = HIDDEN_DIM):
        super().__init__()
        # shared feature layers
        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
        )
        # value stream  V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
        # advantage stream  A(s, a)
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        features  = self.feature(x)
        value     = self.value_stream(features)       # (batch, 1)
        advantage = self.advantage_stream(features)   # (batch, action_dim)
        # Q = V + (A - mean(A))  for identifiability
        q_values  = value + advantage - advantage.mean(dim=1, keepdim=True)
        return q_values


# Verify shapes
_test_state = torch.randn(1, 4).to(device)
_dqn = DQNNetwork(4, 2).to(device)
_dueling = DuelingDQNNetwork(4, 2).to(device)
print(f"DQN output shape:     {_dqn(_test_state).shape}")       # (1, 2)
print(f"Dueling output shape: {_dueling(_test_state).shape}")   # (1, 2)

---
## 8. DQN Agent

A unified agent class that supports **DQN**, **Double DQN**, and **Dueling DQN**
through the `variant` parameter.

In [ ]:
class DQNAgent:
    """
    Deep Q-Network agent supporting three variants:
      - 'dqn'      : standard DQN
      - 'double'    : Double DQN (same network, different target computation)
      - 'dueling'   : Dueling DQN (different network architecture)
    """

    def __init__(self, state_dim: int, action_dim: int, variant: str = 'dqn',
                 hidden_dim: int = HIDDEN_DIM, lr: float = LR,
                 gamma: float = GAMMA, buffer_size: int = BUFFER_SIZE,
                 batch_size: int = BATCH_SIZE, target_update: int = TARGET_UPDATE):

        self.action_dim    = action_dim
        self.variant       = variant
        self.gamma         = gamma
        self.batch_size    = batch_size
        self.target_update = target_update

        # choose network class
        NetClass = DuelingDQNNetwork if variant == 'dueling' else DQNNetwork

        self.policy_net = NetClass(state_dim, action_dim, hidden_dim).to(device)
        self.target_net = NetClass(state_dim, action_dim, hidden_dim).to(device)
        self.update_target()  # initialise target = policy

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(buffer_size)

    # ---- action selection with epsilon-greedy ----
    def select_action(self, state, epsilon: float) -> int:
        if random.random() < epsilon:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.policy_net(state_t)
            return q_values.argmax(dim=1).item()

    # ---- single gradient step on a mini-batch ----
    def train_step(self) -> float:
        if len(self.replay_buffer) < self.batch_size:
            return 0.0

        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)

        # current Q-values for chosen actions
        q_values = self.policy_net(states).gather(1, actions)  # (batch, 1)

        with torch.no_grad():
            if self.variant == 'double' or self.variant == 'dueling':
                # Double DQN target (also used for Dueling to reduce overestimation)
                # select best action with ONLINE network
                best_actions = self.policy_net(next_states).argmax(dim=1, keepdim=True)
                # evaluate with TARGET network
                next_q = self.target_net(next_states).gather(1, best_actions)
            else:
                # Standard DQN target
                next_q = self.target_net(next_states).max(dim=1, keepdim=True)[0]

            target = rewards + self.gamma * next_q * (1.0 - dones)

        loss = nn.MSELoss()(q_values, target)

        self.optimizer.zero_grad()
        loss.backward()
        # gradient clipping for stability
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=1.0)
        self.optimizer.step()

        return loss.item()

    # ---- hard update: copy policy -> target ----
    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    # ---- full training loop ----
    def train(self, env, n_episodes: int = N_EPISODES,
              eps_start: float = EPS_START, eps_end: float = EPS_END,
              eps_decay: float = EPS_DECAY, verbose: bool = True):

        episode_rewards = []
        losses = []
        epsilons = []
        q_value_history = []  # track Q-value statistics
        epsilon = eps_start

        for ep in range(1, n_episodes + 1):
            state, _ = env.reset(seed=SEED + ep)
            total_reward = 0.0
            ep_losses = []

            done = False
            while not done:
                action = self.select_action(state, epsilon)
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated

                self.replay_buffer.push(state, action, reward, next_state, float(done))
                loss = self.train_step()
                if loss > 0:
                    ep_losses.append(loss)

                state = next_state
                total_reward += reward

            # epsilon decay
            epsilon = max(eps_end, epsilon * eps_decay)
            epsilons.append(epsilon)

            # periodic target update
            if ep % self.target_update == 0:
                self.update_target()

            episode_rewards.append(total_reward)
            if ep_losses:
                losses.append(np.mean(ep_losses))
            else:
                losses.append(0.0)

            # record Q-value stats every 50 episodes
            if ep % 50 == 0:
                with torch.no_grad():
                    sample_state = torch.FloatTensor(state).unsqueeze(0).to(device)
                    qv = self.policy_net(sample_state).cpu().numpy().flatten()
                    q_value_history.append({'episode': ep, 'q_values': qv.tolist(),
                                           'mean': float(qv.mean()), 'max': float(qv.max())})

            # logging
            if verbose and ep % 50 == 0:
                avg_100 = np.mean(episode_rewards[-100:])
                print(f"  Ep {ep:4d} | Reward {total_reward:6.1f} | "
                      f"Avg100 {avg_100:6.1f} | eps {epsilon:.3f} | Loss {losses[-1]:.4f}")

        return episode_rewards, losses, epsilons, q_value_history


print("DQNAgent class defined.")

---
## 9. Helper Utilities

In [ ]:
def make_env(seed: int = SEED):
    """Create a seeded CartPole-v1 environment."""
    env = gym.make('CartPole-v1')
    env.reset(seed=seed)
    return env


def smooth(values, window: int = 20):
    """Simple moving average for smoothing curves."""
    if len(values) < window:
        return values
    cumsum = np.cumsum(np.insert(values, 0, 0))
    return (cumsum[window:] - cumsum[:-window]) / window


def episodes_to_solve(rewards, threshold: float = 195.0, window: int = 100):
    """Return the first episode where the running average >= threshold, or -1."""
    for i in range(window, len(rewards) + 1):
        if np.mean(rewards[i - window:i]) >= threshold:
            return i
    return -1


print("Utilities defined.")

---
## 10. Train Standard DQN

In [ ]:
print("="*60)
print("Training: Standard DQN on CartPole-v1")
print("="*60)

# reset seeds for fair comparison
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

env_dqn = make_env(SEED)
agent_dqn = DQNAgent(state_dim=4, action_dim=2, variant='dqn')
rewards_dqn, losses_dqn, epsilons_dqn, qhist_dqn = agent_dqn.train(env_dqn, N_EPISODES)
env_dqn.close()

solve_ep_dqn = episodes_to_solve(rewards_dqn)
final_avg_dqn = np.mean(rewards_dqn[-100:])
print(f"\nDQN solved at episode: {solve_ep_dqn}")
print(f"Final avg reward (last 100): {final_avg_dqn:.1f}")

---
## 11. Train Double DQN

In [ ]:
print("="*60)
print("Training: Double DQN on CartPole-v1")
print("="*60)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

env_ddqn = make_env(SEED)
agent_ddqn = DQNAgent(state_dim=4, action_dim=2, variant='double')
rewards_ddqn, losses_ddqn, epsilons_ddqn, qhist_ddqn = agent_ddqn.train(env_ddqn, N_EPISODES)
env_ddqn.close()

solve_ep_ddqn = episodes_to_solve(rewards_ddqn)
final_avg_ddqn = np.mean(rewards_ddqn[-100:])
print(f"\nDouble DQN solved at episode: {solve_ep_ddqn}")
print(f"Final avg reward (last 100): {final_avg_ddqn:.1f}")

---
## 12. Train Dueling DQN

In [ ]:
print("="*60)
print("Training: Dueling DQN on CartPole-v1")
print("="*60)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

env_duel = make_env(SEED)
agent_duel = DQNAgent(state_dim=4, action_dim=2, variant='dueling')
rewards_duel, losses_duel, epsilons_duel, qhist_duel = agent_duel.train(env_duel, N_EPISODES)
env_duel.close()

solve_ep_duel = episodes_to_solve(rewards_duel)
final_avg_duel = np.mean(rewards_duel[-100:])
print(f"\nDueling DQN solved at episode: {solve_ep_duel}")
print(f"Final avg reward (last 100): {final_avg_duel:.1f}")

---
## 13. Visualization: Training Reward Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- raw rewards (faded) + smoothed ---
ax = axes[0]
for rewards, label, color in [
    (rewards_dqn, 'DQN', COLORS[0]),
    (rewards_ddqn, 'Double DQN', COLORS[1]),
    (rewards_duel, 'Dueling DQN', COLORS[2]),
]:
    ax.plot(rewards, alpha=0.15, color=color)
    smoothed = smooth(rewards, window=20)
    ax.plot(range(20 - 1, 20 - 1 + len(smoothed)), smoothed, label=label, color=color)

ax.axhline(195, color='grey', linestyle='--', alpha=0.6, label='Solved (195)')
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward')
ax.set_title('Training Reward Curves (Smoothed)')
ax.legend(loc='lower right')

# --- 100-episode running average ---
ax = axes[1]
for rewards, label, color in [
    (rewards_dqn, 'DQN', COLORS[0]),
    (rewards_ddqn, 'Double DQN', COLORS[1]),
    (rewards_duel, 'Dueling DQN', COLORS[2]),
]:
    running = smooth(rewards, window=100)
    ax.plot(range(99, 99 + len(running)), running, label=label, color=color)

ax.axhline(195, color='grey', linestyle='--', alpha=0.6, label='Solved (195)')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg Reward (100-ep window)')
ax.set_title('100-Episode Running Average')
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

---
## 14. Visualization: Epsilon Decay Schedule

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(epsilons_dqn, color=COLORS[3], label='Epsilon (multiplicative decay)')
ax.axhline(EPS_END, color='grey', linestyle='--', alpha=0.6, label=f'eps_end = {EPS_END}')
ax.fill_between(range(len(epsilons_dqn)), epsilons_dqn, alpha=0.15, color=COLORS[3])
ax.set_xlabel('Episode')
ax.set_ylabel('Epsilon')
ax.set_title(f'Epsilon-Greedy Decay Schedule (decay={EPS_DECAY})')
ax.legend()
plt.tight_layout()
plt.show()

# find when epsilon effectively reaches minimum
eps_min_ep = next((i for i, e in enumerate(epsilons_dqn) if e <= EPS_END + 0.001), len(epsilons_dqn))
print(f"Epsilon reaches ~{EPS_END} around episode {eps_min_ep}")

---
## 15. Visualization: Training Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for losses, label, color in [
    (losses_dqn, 'DQN', COLORS[0]),
    (losses_ddqn, 'Double DQN', COLORS[1]),
    (losses_duel, 'Dueling DQN', COLORS[2]),
]:
    smoothed = smooth(losses, window=20)
    ax.plot(range(19, 19 + len(smoothed)), smoothed, label=label, color=color, alpha=0.85)

ax.set_xlabel('Episode')
ax.set_ylabel('Mean Episode Loss')
ax.set_title('Training Loss Curves (Smoothed)')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

---
## 16. Visualization: Q-Value Distributions

In [ ]:
def collect_q_values(agent, env, n_episodes: int = 10):
    """Collect Q-values over several episodes for analysis."""
    all_q = []
    for ep in range(n_episodes):
        state, _ = env.reset(seed=SEED + 1000 + ep)
        done = False
        while not done:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                q = agent.policy_net(state_t).cpu().numpy().flatten()
                all_q.append(q)
            action = q.argmax()
            state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
    return np.array(all_q)


# collect Q-values from trained agents
env_eval = make_env(SEED)
q_dqn   = collect_q_values(agent_dqn, env_eval)
q_ddqn  = collect_q_values(agent_ddqn, env_eval)
q_duel  = collect_q_values(agent_duel, env_eval)
env_eval.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, q_vals, label, color in [
    (axes[0], q_dqn, 'DQN', COLORS[0]),
    (axes[1], q_ddqn, 'Double DQN', COLORS[1]),
    (axes[2], q_duel, 'Dueling DQN', COLORS[2]),
]:
    ax.hist(q_vals[:, 0], bins=40, alpha=0.6, label='Q(s, left)', color=color)
    ax.hist(q_vals[:, 1], bins=40, alpha=0.6, label='Q(s, right)', color=COLORS[3])
    ax.set_title(f'{label} Q-Value Distribution')
    ax.set_xlabel('Q-value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"DQN     mean Q: {q_dqn.mean():.2f}, max Q: {q_dqn.max():.2f}")
print(f"DDQN    mean Q: {q_ddqn.mean():.2f}, max Q: {q_ddqn.max():.2f}")
print(f"Dueling mean Q: {q_duel.mean():.2f}, max Q: {q_duel.max():.2f}")

---
## 17. Visualization: Replay Buffer Analysis

In [ ]:
def analyze_buffer(agent, name: str):
    """Analyze the contents of a replay buffer."""
    buffer = agent.replay_buffer
    rewards_buf = [t.reward for t in buffer.buffer]
    dones_buf   = [t.done for t in buffer.buffer]
    actions_buf = [t.action for t in buffer.buffer]
    return {
        'name': name,
        'size': len(buffer),
        'rewards': rewards_buf,
        'done_rate': np.mean(dones_buf),
        'action_dist': [actions_buf.count(a) / len(actions_buf) for a in range(agent.action_dim)],
    }


buf_stats = [
    analyze_buffer(agent_dqn, 'DQN'),
    analyze_buffer(agent_ddqn, 'Double DQN'),
    analyze_buffer(agent_duel, 'Dueling DQN'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# reward distributions in buffer
ax = axes[0]
for i, bs in enumerate(buf_stats):
    ax.hist(bs['rewards'], bins=30, alpha=0.5, label=bs['name'], color=COLORS[i])
ax.set_title('Replay Buffer: Reward Distribution')
ax.set_xlabel('Reward')
ax.set_ylabel('Count')
ax.legend(fontsize=9)

# action distributions
ax = axes[1]
x = np.arange(2)
width = 0.25
for i, bs in enumerate(buf_stats):
    ax.bar(x + i * width, bs['action_dist'], width, label=bs['name'], color=COLORS[i])
ax.set_title('Replay Buffer: Action Distribution')
ax.set_xlabel('Action')
ax.set_ylabel('Proportion')
ax.set_xticks(x + width)
ax.set_xticklabels(['Left (0)', 'Right (1)'])
ax.legend(fontsize=9)

# buffer sizes and done rates
ax = axes[2]
names = [bs['name'] for bs in buf_stats]
sizes = [bs['size'] for bs in buf_stats]
done_rates = [bs['done_rate'] * 100 for bs in buf_stats]
ax.bar(names, sizes, color=[COLORS[i] for i in range(3)], alpha=0.7)
ax.set_ylabel('Buffer Size')
ax.set_title('Buffer Size & Terminal Rate')
# add done-rate text
for i, (s, d) in enumerate(zip(sizes, done_rates)):
    ax.text(i, s + 50, f'Done: {d:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 18. Visualization: Performance Comparison

In [ ]:
variants = ['DQN', 'Double DQN', 'Dueling DQN']
solve_episodes = [solve_ep_dqn, solve_ep_ddqn, solve_ep_duel]
final_avgs = [final_avg_dqn, final_avg_ddqn, final_avg_duel]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# episodes to solve
ax = axes[0]
bars = ax.bar(variants, [e if e > 0 else N_EPISODES for e in solve_episodes],
              color=[COLORS[0], COLORS[1], COLORS[2]], alpha=0.8)
ax.set_ylabel('Episodes to Solve')
ax.set_title('Episodes to Solve CartPole-v1 (avg reward >= 195)')
for bar, ep in zip(bars, solve_episodes):
    label = str(ep) if ep > 0 else 'Not solved'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            label, ha='center', fontsize=11, fontweight='bold')

# final average reward
ax = axes[1]
bars = ax.bar(variants, final_avgs,
              color=[COLORS[0], COLORS[1], COLORS[2]], alpha=0.8)
ax.axhline(195, color='grey', linestyle='--', alpha=0.6, label='Solved threshold')
ax.set_ylabel('Avg Reward (Last 100 Episodes)')
ax.set_title('Final Performance Comparison')
ax.legend()
for bar, avg in zip(bars, final_avgs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{avg:.1f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 19. Q-Value Evolution During Training

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, qhist, label, color in [
    (axes[0], qhist_dqn, 'DQN', COLORS[0]),
    (axes[1], qhist_ddqn, 'Double DQN', COLORS[1]),
    (axes[2], qhist_duel, 'Dueling DQN', COLORS[2]),
]:
    if qhist:
        eps = [h['episode'] for h in qhist]
        means = [h['mean'] for h in qhist]
        maxes = [h['max'] for h in qhist]
        ax.plot(eps, means, 'o-', color=color, label='Mean Q')
        ax.plot(eps, maxes, 's--', color=COLORS[3], label='Max Q')
        ax.fill_between(eps, means, maxes, alpha=0.15, color=color)
    ax.set_title(f'{label}: Q-Value Evolution')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Q-value')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 20. Target Network Impact Analysis

We compare Q-value stability with and without the target network by examining
how much Q-values fluctuate across consecutive training checkpoints.

In [ ]:
# Train a small DQN WITHOUT target network (target updated every step) for comparison
print("Training DQN with frequent target updates (every 1 episode) ...")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

env_notarget = make_env(SEED)
agent_notarget = DQNAgent(state_dim=4, action_dim=2, variant='dqn', target_update=1)
rewards_notarget, losses_notarget, _, qhist_notarget = agent_notarget.train(
    env_notarget, n_episodes=300, verbose=True
)
env_notarget.close()

# Compare Q-value oscillations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# reward curves
ax = axes[0]
ax.plot(smooth(rewards_dqn[:300], 20), label='DQN (target update=10)', color=COLORS[0])
ax.plot(smooth(rewards_notarget, 20), label='DQN (target update=1)', color=COLORS[1], linestyle='--')
ax.axhline(195, color='grey', linestyle='--', alpha=0.4)
ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed Reward')
ax.set_title('Impact of Target Network Update Frequency')
ax.legend()

# loss oscillations
ax = axes[1]
ax.plot(smooth(losses_dqn[:300], 20), label='DQN (target update=10)', color=COLORS[0])
ax.plot(smooth(losses_notarget, 20), label='DQN (target update=1)', color=COLORS[1], linestyle='--')
ax.set_xlabel('Episode')
ax.set_ylabel('Smoothed Loss')
ax.set_title('Loss Oscillation Comparison')
ax.set_yscale('log')
ax.legend()

plt.tight_layout()
plt.show()

# compute oscillation metric: std of consecutive loss differences
def loss_oscillation(losses):
    diffs = np.diff(losses)
    return np.std(diffs)

osc_target = loss_oscillation(losses_dqn[:300])
osc_notarget = loss_oscillation(losses_notarget)
print(f"\nLoss oscillation (std of diffs):")
print(f"  Target update=10: {osc_target:.4f}")
print(f"  Target update=1:  {osc_notarget:.4f}")
print(f"  Target network reduces oscillation: {osc_target < osc_notarget}")

---
## 21. Dueling Architecture Internals

Let's peek inside the trained Dueling DQN and see how it decomposes
Q-values into value and advantage components.

In [ ]:
# Collect V(s) and A(s,a) from the Dueling network for sample states
env_vis = make_env(SEED)
states_vis = []
state, _ = env_vis.reset(seed=SEED + 999)
done = False
while not done and len(states_vis) < 200:
    states_vis.append(state)
    action = agent_duel.select_action(state, epsilon=0.0)
    state, _, terminated, truncated, _ = env_vis.step(action)
    done = terminated or truncated
env_vis.close()

states_t = torch.FloatTensor(np.array(states_vis)).to(device)
with torch.no_grad():
    features   = agent_duel.policy_net.feature(states_t)
    values     = agent_duel.policy_net.value_stream(features).cpu().numpy().flatten()
    advantages = agent_duel.policy_net.advantage_stream(features).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(values, color=COLORS[2], label='V(s)')
ax.set_title('Dueling DQN: Value Stream V(s)')
ax.set_xlabel('Step')
ax.set_ylabel('Value')
ax.legend()

ax = axes[1]
ax.plot(advantages[:, 0], color=COLORS[0], label='A(s, left)', alpha=0.8)
ax.plot(advantages[:, 1], color=COLORS[1], label='A(s, right)', alpha=0.8)
ax.set_title('Dueling DQN: Advantage Stream A(s,a)')
ax.set_xlabel('Step')
ax.set_ylabel('Advantage')
ax.legend()

ax = axes[2]
q_combined = values[:, None] + advantages - advantages.mean(axis=1, keepdims=True)
ax.plot(q_combined[:, 0], color=COLORS[0], label='Q(s, left)', alpha=0.8)
ax.plot(q_combined[:, 1], color=COLORS[1], label='Q(s, right)', alpha=0.8)
ax.set_title('Dueling DQN: Combined Q(s,a)')
ax.set_xlabel('Step')
ax.set_ylabel('Q-value')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Value stream range:     [{values.min():.2f}, {values.max():.2f}]")
print(f"Advantage stream range: [{advantages.min():.2f}, {advantages.max():.2f}]")
print(f"Advantage mean (should be ~0): {advantages.mean():.4f}")

---
## 22. Evaluation: Greedy Policy Rollouts

In [ ]:
def evaluate_agent(agent, n_eval: int = 50, seed_offset: int = 5000):
    """Run greedy rollouts and return per-episode rewards."""
    env = make_env(SEED)
    eval_rewards = []
    for ep in range(n_eval):
        state, _ = env.reset(seed=SEED + seed_offset + ep)
        total = 0.0
        done = False
        while not done:
            action = agent.select_action(state, epsilon=0.0)  # greedy
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total += reward
        eval_rewards.append(total)
    env.close()
    return eval_rewards


eval_dqn  = evaluate_agent(agent_dqn)
eval_ddqn = evaluate_agent(agent_ddqn)
eval_duel = evaluate_agent(agent_duel)

fig, ax = plt.subplots(figsize=(10, 5))
data = [eval_dqn, eval_ddqn, eval_duel]
bp = ax.boxplot(data, labels=variants, patch_artist=True)
for patch, color in zip(bp['boxes'], COLORS[:3]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.axhline(195, color='grey', linestyle='--', alpha=0.6, label='Solved (195)')
ax.set_ylabel('Episode Reward')
ax.set_title('Greedy Policy Evaluation (50 Episodes Each)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Evaluation results (mean +/- std over 50 episodes):")
for name, ev in zip(variants, data):
    print(f"  {name:15s}: {np.mean(ev):.1f} +/- {np.std(ev):.1f}")

---
## 23. Overestimation Bias Analysis

One of the key motivations for Double DQN is reducing overestimation bias.
Let's compare the max Q-values predicted by each variant.

In [ ]:
# Compare max Q-values across variants for the same states
env_bias = make_env(SEED)
sample_states = []
for ep in range(20):
    state, _ = env_bias.reset(seed=SEED + 2000 + ep)
    done = False
    while not done:
        sample_states.append(state)
        action = random.randrange(2)
        state, _, terminated, truncated, _ = env_bias.step(action)
        done = terminated or truncated
env_bias.close()

sample_t = torch.FloatTensor(np.array(sample_states)).to(device)

with torch.no_grad():
    maxq_dqn  = agent_dqn.policy_net(sample_t).max(dim=1)[0].cpu().numpy()
    maxq_ddqn = agent_ddqn.policy_net(sample_t).max(dim=1)[0].cpu().numpy()
    maxq_duel = agent_duel.policy_net(sample_t).max(dim=1)[0].cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(maxq_dqn, bins=40, alpha=0.5, label=f'DQN (mean={maxq_dqn.mean():.2f})', color=COLORS[0])
ax.hist(maxq_ddqn, bins=40, alpha=0.5, label=f'Double DQN (mean={maxq_ddqn.mean():.2f})', color=COLORS[1])
ax.hist(maxq_duel, bins=40, alpha=0.5, label=f'Dueling DQN (mean={maxq_duel.mean():.2f})', color=COLORS[2])
ax.set_xlabel('Max Q-value')
ax.set_ylabel('Count')
ax.set_title('Distribution of Max Q-values Across Random States')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Max Q-value statistics:")
print(f"  DQN:        mean={maxq_dqn.mean():.2f}, std={maxq_dqn.std():.2f}")
print(f"  Double DQN: mean={maxq_ddqn.mean():.2f}, std={maxq_ddqn.std():.2f}")
print(f"  Dueling:    mean={maxq_duel.mean():.2f}, std={maxq_duel.std():.2f}")

---
## 24. Network Parameter Statistics

In [ ]:
def count_parameters(model):
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def param_stats(model, name):
    """Print parameter statistics for a model."""
    total = count_parameters(model)
    print(f"\n{name} ({total:,} parameters):")
    for pname, param in model.named_parameters():
        print(f"  {pname:30s} shape={str(list(param.shape)):15s} "
              f"mean={param.data.mean():.4f}  std={param.data.std():.4f}")


param_stats(agent_dqn.policy_net, 'DQN')
param_stats(agent_duel.policy_net, 'Dueling DQN')

print(f"\nParameter count comparison:")
print(f"  DQN:     {count_parameters(agent_dqn.policy_net):,}")
print(f"  Dueling: {count_parameters(agent_duel.policy_net):,}")

---
## 25. Summary Table

In [ ]:
print("\n" + "="*80)
print(f"{'Variant':15s} | {'Solved At':>10s} | {'Final Avg':>10s} | {'Eval Mean':>10s} | {'Params':>8s}")
print("-"*80)

results = [
    ('DQN', solve_ep_dqn, final_avg_dqn, np.mean(eval_dqn),
     count_parameters(agent_dqn.policy_net)),
    ('Double DQN', solve_ep_ddqn, final_avg_ddqn, np.mean(eval_ddqn),
     count_parameters(agent_ddqn.policy_net)),
    ('Dueling DQN', solve_ep_duel, final_avg_duel, np.mean(eval_duel),
     count_parameters(agent_duel.policy_net)),
]

for name, solved, favg, emean, params in results:
    solved_str = str(solved) if solved > 0 else 'N/A'
    print(f"{name:15s} | {solved_str:>10s} | {favg:>10.1f} | {emean:>10.1f} | {params:>8,}")

print("="*80)

---
## 26. Verification Tests

In [ ]:
print("="*60)
print("VERIFICATION")
print("="*60)

# Test 1: DQN solves CartPole
dqn_solved = final_avg_dqn >= 195.0
status = "[PASS]" if dqn_solved else "[FAIL]"
print(f"{status} DQN solves CartPole (avg reward >= 195): {final_avg_dqn:.1f}")

# Test 2: Double DQN solves CartPole
ddqn_solved = final_avg_ddqn >= 195.0
status = "[PASS]" if ddqn_solved else "[FAIL]"
print(f"{status} Double DQN solves CartPole (avg reward >= 195): {final_avg_ddqn:.1f}")

# Test 3: Dueling DQN solves CartPole
duel_solved = final_avg_duel >= 195.0
status = "[PASS]" if duel_solved else "[FAIL]"
print(f"{status} Dueling DQN solves CartPole (avg reward >= 195): {final_avg_duel:.1f}")

# Test 4: Target network reduces Q-value oscillation
target_helps = osc_target < osc_notarget
status = "[PASS]" if target_helps else "[FAIL]"
print(f"{status} Target network reduces Q-value oscillation: "
      f"{osc_target:.4f} < {osc_notarget:.4f} = {target_helps}")

# Test 5: Replay buffer maintains correct size
buf_correct = (
    len(agent_dqn.replay_buffer) <= BUFFER_SIZE and
    len(agent_ddqn.replay_buffer) <= BUFFER_SIZE and
    len(agent_duel.replay_buffer) <= BUFFER_SIZE
)
status = "[PASS]" if buf_correct else "[FAIL]"
print(f"{status} Replay buffer maintains correct size (<= {BUFFER_SIZE}): "
      f"DQN={len(agent_dqn.replay_buffer)}, "
      f"DDQN={len(agent_ddqn.replay_buffer)}, "
      f"Dueling={len(agent_duel.replay_buffer)}")

print("\n" + "="*60)
n_pass = sum([dqn_solved, ddqn_solved, duel_solved, target_helps, buf_correct])
print(f"Results: {n_pass}/5 tests passed")
print("="*60)

---
## 27. Key Takeaways

1. **Experience replay** breaks temporal correlations and allows sample reuse, dramatically
   improving sample efficiency and training stability.

2. **Target networks** stabilise training by providing a slowly-moving target for the Bellman
   equation. Without them, the moving-target problem causes oscillations and divergence.

3. **Double DQN** is a near-zero-cost improvement: same network architecture, just a different
   way of computing the target. It reduces the systematic overestimation bias of standard DQN.

4. **Dueling DQN** factorises Q-values into state value $V(s)$ and action advantages $A(s,a)$.
   This is especially beneficial when many actions have similar values -- the value stream
   learns the state quality without needing to evaluate every action.

5. The **epsilon-greedy** schedule balances exploration and exploitation. A multiplicative decay
   of $0.995$ per episode provides a smooth transition from random to greedy behaviour.

6. All three variants reliably solve CartPole-v1 within 500 episodes, but Double and Dueling
   DQN often converge faster and more stably.

**Next steps:** Prioritised Experience Replay (PER), Noisy Networks for exploration,
distributional RL (C51, QR-DQN), and the Rainbow agent that combines all improvements.